In [2]:
from numpy import dtype
from sklearn.datasets import fetch_openml
mnist = fetch_openml('mnist_784', version=1)
print(mnist.keys())

dict_keys(['data', 'target', 'frame', 'categories', 'feature_names', 'target_names', 'DESCR', 'details', 'url'])


In [3]:
X, y = mnist['data'], mnist['target']
print(X.shape)  
print(y.shape)

(70000, 784)
(70000,)


In [4]:
import numpy as np
y = y.astype(np.uint8)
X_train, X_test, Y_train, Y_test = X[:60000], X[60000:], y[:60000], y[60000:] 

1. Spróbuj stworzyć klasyfikator zbioru danych MNIST osiągający ponad 97% dokładności dla zbio
ru testowego. Podpowiedź: całkiem nieźle sprawdza się klasyfikator KNeighborsClassifier;
musisz tylko dobrać odpowiednie wartości hiperparametrów (zastosuj metodę przeszukiwa
nia siatki wobec hiperparametrów weights i n_neighbors)

In [4]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV
param_grid = [{"weights": ["uniform", "distance"],"n_neighbors" : [2,3,4,5]}]
KN_clf = KNeighborsClassifier()
grid_serch = GridSearchCV(KN_clf, param_grid, cv = 3, scoring='accuracy')
grid_serch.fit(X_train, Y_train)
best_model = grid_serch.best_estimator_
acuracy = best_model.score(X_test, Y_test)
print(acuracy)

0.9714


2.Napisz funkcję przesuwającą obraz z zestawu danych MNIST o jeden piksel w dowolnym kie
runku (w lewo, prawo, do góry lub w dół)5. Następnie dla każdego obrazu w zbiorze uczącym
stwórz cztery przesunięte kopie (po jednej na każdy kierunek przesunięcia) i dołącz je do zbioru
uczącego. Na koniec wytrenuj swój najlepszy model wobec takiego rozszerzonego zestawu
danych i zmierz jego dokładność przy użyciu zbioru testowego. Twój model powinien teraz
sprawować się jeszcze lepiej! Taka technika sztucznego powiększania zestawu uczącego bywa na
zywana dogenerowaniem danych (ang. data augmentation) lub rozszerzeniem zbioru uczącego
(ang. training set expansion).

In [1]:
import numpy as np
from scipy.ndimage.interpolation import shift 
def przesunięcie(X_train, Y_train):
    kierunki = [[1,0],[-1,0],[0,1],[0,-1]]
    X_aug = []
    Y_aug = []
    for obraz, etykieta in zip(X_train.to_numpy(), Y_train):
        X_aug.append(obraz)
        Y_aug.append(etykieta)
        for dx, dy in kierunki:
            img = obraz.reshape(28,28)
            new_img = shift(img,[dx,dy], cval = 0)
            X_aug.append(new_img.reshape(784))
            Y_aug.append(etykieta)
    return X_aug, Y_aug
    

C:\Users\wikto\AppData\Local\Temp\ipykernel_24376\563507683.py:2: DeprecationWarning: Please import `shift` from the `scipy.ndimage` namespace; the `scipy.ndimage.interpolation` namespace is deprecated and will be removed in SciPy 2.0.0.
  from scipy.ndimage.interpolation import shift


In [5]:
przesuniete = przesunięcie(X_train, Y_train)

In [10]:
X_small = przesuniete[0][:80000]
Y_small = przesuniete[1][:80000]
param_grid = [{"weights": ["uniform", "distance"],"n_neighbors" : [2,3,4,5]}]
KN_clf = KNeighborsClassifier()
grid_search = GridSearchCV(KN_clf, param_grid, cv = 3, scoring='accuracy')
grid_search.fit(X_small, Y_small)
best_model = grid_search.best_estimator_
acuracy = best_model.score(X_test, Y_test)
print(acuracy)

C:\Users\wikto\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2684: UserWarning: X has feature names, but KNeighborsClassifier was fitted without feature names
  warnings.warn(


0.966
